# *Drosophila* pipeline — Figs. 4 & 5 / Supp. Figs. 7–11

This notebook applies the multi-timescale transfer-operator pipeline of
**Kaur, Jain, & Berman (2026)**, *"Using timescale as a state coordinate
reveals the metastable geometry of behavior,"* to freely-moving
*Drosophila*, recovering four metastable behavioral basins — *Idle & Slow*,
*Anterior Movements*, *Posterior & Wing Movements*, and *Locomotion* — as
four linear arms radiating from a hub in eigenvector space.  The four-arm
geometry at M = 4 is the central methodological claim of the paper.

It runs end-to-end from the cached artifacts shipped in `data/flies/`
(multi- and fixed-timescale cluster sequences, G-PCCA memberships, the
behavior-map density grid, and the slow-mode dynamics).  The raw
joint-angle → wavelet → PCA upstream is documented but gated behind
`RECOMPUTE_UPSTREAM`, since those intermediates are large and not shipped.

In [ ]:
# --- Colab / local setup ---
# If you're running on Colab, uncomment the next lines to install the pip
# dependencies that aren't in the default Colab image and to clone this repo
# (for the helper modules + cached data in data/flies/).
#
# !pip install pygpcca powerlaw umap-learn
# !git clone https://github.com/bermanlabemory/slowmode.git
# %cd slowmode

import os, sys, pickle, time
import numpy as np
import matplotlib.pyplot as plt

# Ensure slowmode/ is on the path.
sys.path.insert(0, os.path.abspath('.'))

import pipeline as pp
import gpcca_utils as gu
import figures as fg

fg.setup_style()
os.makedirs('outputs', exist_ok=True)

In [ ]:
# --- imports specific to this notebook ---
from sklearn.decomposition import PCA
import warnings; warnings.filterwarnings('ignore', category=UserWarning)


## 1. Load the fly wavelet-PC, behavior-map, and cluster sequences

`flies_wlets_pca.pkl` is a list of 30 flies, each a $(360001, 15)$ array of
wavelet PCA projections.

`flies_zvals.pkl` is a list of 30 $(360001, 2)$ arrays giving the Berman
2014 behavior-map coordinates (used for Fig. 5A).

`flies_state_partitions.pkl` is a $\{fly \to \text{int array of length}~ 359990\}$
dict of cached cluster sequences from $k$-means at $N = 1000$.


In [ ]:
# Load the shipped, precomputed fly data.  Everything needed for Figs. 4 & 5
# and Supp. Figs. S7-S11 ships in data/flies/.
DATA_DIR = 'data/flies'

# Multi-timescale cluster sequences (one integer label per frame, per fly),
# N = 1000 clusters.
with open(f'{DATA_DIR}/states_flies.pkl', 'rb') as f:
    states_per_fly = {i: np.asarray(s, dtype=int)
                      for i, s in pickle.load(f).items()}
n_flies = len(states_per_fly)
print(f'{n_flies} flies; example sequence length = '
      f'{len(next(iter(states_per_fly.values())))}')

# --- Upstream (raw joint angles -> wavelet -> PCA) ------------------------
# The wavelet-PC intermediates (~600 MB) and the behavior-map coordinates
# (zvals) are large and are NOT shipped.  To rebuild the cluster sequences
# from raw joint angles set RECOMPUTE_UPSTREAM = True and supply the
# joint-angle time series; the recipe is pp.morlet_wavelet_amplitudes ->
# pp.pca_with_shuffle_threshold -> pp.delay_embed -> pp.kmeans_partition.
# By default we use the shipped cluster sequences loaded above.
RECOMPUTE_UPSTREAM = False

## 2. Pool cluster sequences and build $T(\tau = 2~\mathrm{s})$


In [ ]:
fs = 100.0
tau_seconds = 2.0
lag = int(tau_seconds * fs)            # 200 frames

all_states = np.concatenate([states_per_fly[i].astype(int)
                              for i in sorted(states_per_fly)])
N = int(all_states.max()) + 1
print(f'pooled states: T = {len(all_states):,}    N = {N}    lag = {lag}')

T_multi = pp.make_transition_matrix(all_states, lag=lag, n_states=N)
pi_multi = pp.stationary_distribution(T_multi)
evals_multi, evecs_multi = pp.leading_eigvecs(T_multi, k=10)
phi_mt = evecs_multi[:, :3].real     # (N, 3) for the eigenspace plots
print(f'leading |lambda_k| = {np.abs(evals_multi)[:6].round(4)}')

M_pick, gap, ratios = gu.select_M_spectral_gap(evals_multi, M_min=2, M_max=8)
print(f'spectral gap selects M = {M_pick} (gap ratio = {gap:.3f})')


## 3. G-PCCA at $M = 4$


In [ ]:
out = gu.run_gpcca(T_multi, M=4, eta=pi_multi)
print(f'crispness    = {out["crispness"]:.3f}')
print(f'basin counts = {out["basin_counts"]}')
print(f'pi per basin = {out["pi_basin"].round(3)}')

# G-PCCA returns basins up to a column permutation.  Adopt the cached
# memberships (same clusters, fixed labeling) so the arm indices match the
# published Fig. 4/5 panels and the behavior-map density grid.
gp_cached = np.load(f'{DATA_DIR}/gpcca_flies_M4_tau2s.npz')
chi = gp_cached['chi']

# Hub-and-arm geometry in (phi_2, phi_3, phi_4).
geo = gu.compute_hub_arms(phi_mt, pi_multi, chi)
print('arm lengths:', geo['arm_lengths'].round(3))

## 4. Fixed-timescale comparison (Fig. 4A)

Re-run the pipeline without the wavelet step: delay-embed the raw 15-D
wavelet-PCs at $d = 8$ (the value selected by the body-text Lorenz Cao
analogue, since the joint-angle Cao saturation is at $d = 8$ here too).

For speed we use a smaller $N$ for this control panel; the published
analysis uses $N = 1000$.


In [ ]:
# Fixed-timescale baseline.  The paper builds this from the raw joint-angle
# trajectories at N = 3000 clusters, d = 8, tau = 2 s.  We ship the cached
# cluster sequences in data/states_flies_fixed.pkl so reviewers
# don't need the 1.9 GB joint-angle file to reproduce Fig 4A.
with open(f'{DATA_DIR}/states_flies_fixed.pkl', 'rb') as f:
    states_per_fly_fixed = pickle.load(f)

all_states_fix = np.concatenate([states_per_fly_fixed[i].astype(int)
                                  for i in sorted(states_per_fly_fixed)])
N_fix = int(all_states_fix.max()) + 1
print(f'fixed: pooled states T = {len(all_states_fix):,}, N = {N_fix}')

T_fix = pp.make_transition_matrix(all_states_fix, lag=lag, n_states=N_fix)
pi_fix = pp.stationary_distribution(T_fix)
evals_fix, evecs_fix = pp.leading_eigvecs(T_fix, k=10)
phi_fix = evecs_fix[:, :3].real
print(f'fixed |lambda_k| = {np.abs(evals_fix)[:6].round(4)}')
print(f'PR (fixed) = {pp.participation_ratio(phi_fix).round(2)}')
print(f'PR (multi) = {pp.participation_ratio(phi_mt).round(2)}')


## Fig. 4A — fixed-timescale 3D eigenspace, three localization clusters

The fixed-timescale operator places almost all of its slow-mode weight on a
handful of clusters: PR_k = 1, 3, 3.  We highlight the three top-norm
clusters in red/teal/gold.


In [ ]:
# Fig 4A: highlight three localization clusters in the fixed-timescale
# eigenspace.  Paper recipe: sort clusters by L2 norm in (phi_2, phi_3, phi_4),
# walk down and keep the first three whose unit-direction vectors are mutually
# distinct (cosine < 0.9).  These are the localization spikes that give
# PR ~ 1, 3, 3.  Axis convention matches the published figure: x = phi_3,
# y = phi_4, z = phi_2; view = (elev=15, azim=120).
from mpl_toolkits.mplot3d import Axes3D  # noqa

norms_f = np.linalg.norm(phi_fix, axis=1)
order = np.argsort(norms_f)[::-1]
seen_dirs, hi_idx = [], []
for cid in order:
    d_i = phi_fix[cid] / max(np.linalg.norm(phi_fix[cid]), 1e-12)
    if any(abs(d_i @ u) > 0.9 for u in seen_dirs):
        continue
    seen_dirs.append(d_i); hi_idx.append(cid)
    if len(hi_idx) >= 3: break

# Axis permutation: column 1 -> x, column 2 -> y, column 0 -> z.
def xyz_of(p):
    return p[..., 1], p[..., 2], p[..., 0]

fig = plt.figure(figsize=(4.6, 4.0))
ax = fig.add_subplot(111, projection='3d')
fx, fy, fz = xyz_of(phi_fix)
ax.scatter(fx, fy, fz, s=8, c='0.65', alpha=0.5,
            edgecolors='none', depthshade=False)
hi_palette = ['#d7263d', '#1b998b', '#e3a72f']
for k, cid in enumerate(hi_idx):
    px, py, pz = xyz_of(phi_fix[cid])
    ax.scatter([px], [py], [pz], s=180, c=hi_palette[k],
                edgecolors='k', linewidths=0.6, zorder=10)
ax.view_init(elev=15, azim=120)
ax.set_xlabel(r'$\phi_3$'); ax.set_ylabel(r'$\phi_4$'); ax.set_zlabel(r'$\phi_2$')
ax.set_title(f'fixed-timescale eigenspace (PR = {pp.participation_ratio(phi_fix).round(1)})')
plt.tight_layout()
fg.save_panel(fig, 'fig4A_fixed_eigenspace')
plt.show()


### What is an arm? What is the hub?

The four-arm-and-hub geometry of Fig. 4B is the central methodological claim of the paper for *Drosophila*.  Here's what those words mean in our notation.

Each cluster $i$ has a position in the 3-D eigenspace spanned by $(\phi_2, \phi_3, \phi_4)$ — the leading non-trivial right eigenvectors of $T(\tau = 2\text{ s})$.  G-PCCA assigns each cluster a soft membership $\chi_j(i) \in [0, 1]$ in each of $M = 4$ basins.

- **The hub** is the $\pi$-weighted mean of all cluster positions:
  $\boldsymbol{h} = \sum_i \pi_i\, \boldsymbol{\phi}(i) \,/\, \sum_i \pi_i$.

  Operationally it is the centre of mass of the cluster cloud weighted by the stationary distribution.  For a reversible chain it would sit at the origin by orthogonality; for our irreversible $T$ the $\pi$-weighted centroid is the right generalisation.

- **The arm $j$** is the line from the hub to the $\chi$-and-$\pi$-weighted centroid of basin $j$:
  $\boldsymbol{c}_j - \boldsymbol{h}$, where $\boldsymbol{c}_j = \sum_i \chi_j(i)\,\pi_i\,\boldsymbol{\phi}(i) \,/\, \sum_i \chi_j(i)\,\pi_i$.

  Each arm direction $\hat{\boldsymbol{a}}_j = (\boldsymbol{c}_j - \boldsymbol{h}) / \|\boldsymbol{c}_j - \boldsymbol{h}\|$ is the slow coordinate along which membership in basin $j$ grows.

The geometric picture is clean: clusters with $\chi_j \approx 1$ for one $j$ pile up at the *tip* of arm $j$; transitional clusters with no dominant $\chi_j$ pile up near the hub.  A trajectory in time looks like an excursion from the hub out along one arm, a return to the hub, and a departure out a different arm.

**Behavioral labels for the four arms** (Arm 1 = *Idle & Slow*, Arm 2 = *Anterior Movements*, Arm 3 = *Posterior & Wing Movements*, Arm 4 = *Locomotion*) come from projecting the $\chi_j$-weighted basin densities onto the Berman 2014 behavior map — that's Fig. 5A, computed later in this notebook.  The geometry recovered by the operator alone is label-free; the behavior map only enters when we want a name for each arm.

## Fig. 4B — multi-timescale 3D eigenspace with four arms


In [ ]:
fig = plt.figure(figsize=(4.6, 4.0))
ax = fig.add_subplot(111, projection='3d')
fg.plot_eigenspace_3d(
    ax, phi_mt, pi=pi_multi, chi=chi,
    hub=geo['hub'], arm_dirs=geo['arm_dirs'],
    arm_centroids=geo['arm_centroids'],
    palette=fg.ARM_PALETTE,
    view=(15, 120),
    axis_order=(1, 2, 0),
    axis_labels=(r'$\phi_3$', r'$\phi_4$', r'$\phi_2$'))
ax.set_title('multi-timescale eigenspace')
plt.tight_layout()
fg.save_panel(fig, 'fig4B_multi_eigenspace')
plt.show()


## Fig. 4C — eigenvalue spectrum


In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 2.8))
fg.plot_eigenvalue_dots(ax, evals_multi[:10], M=4)
plt.tight_layout()
fg.save_panel(fig, 'fig4C_eigenvalue_spectrum')
plt.show()


## Fig. 4D — representative $\chi_j(t)$ for one fly

Soft basin memberships $\chi_j(t)$ for one representative fly over the
first 20 minutes, with a 5 s rolling smoothing window.  Multiple $\chi_j$
are simultaneously elevated and the dominant arm shifts gradually,
reproducing the continuous, graded organization documented in Berman et
al. (2014, 2016).


In [ ]:
fly_id = 0
states_fly = states_per_fly[fly_id].astype(int)
T_window = 20 * 60 * int(fs)        # 20 minutes
chi_fly = chi[states_fly[:T_window]]
t_fly = np.arange(T_window) / fs

fig, ax = plt.subplots(figsize=(7.2, 2.6))
fg.plot_chi_timeseries(ax, t_fly, chi_fly, palette=fg.ARM_PALETTE,
                        smoothing_window=int(5 * fs),
                        labels=[f'arm {j+1}' for j in range(4)])
ax.set_title(f'representative fly (id = {fly_id})')
plt.tight_layout()
fg.save_panel(fig, 'fig4D_chi_timeseries')
plt.show()


## Fig. 5A — $\chi$-weighted behavior-map enrichment quartet

Per-arm enrichment density on the Berman 2014 behavioral map.  For each arm
$j$ we compute the $\chi_j$-weighted occupancy density on a 2D grid in
$(z_1, z_2)$, then subtract the across-arm mean so that red regions are
enriched and blue depleted.

Each panel is rendered separately (one per arm), in the Okabe-Ito palette
order (idle & slow / anterior / posterior & wing / locomotion gaits).


In [ ]:
# Fig. 5A uses the chi-weighted density on the Berman 2014 behavior map.
# The raw map coordinates (zvals) are large and not shipped, so we load the
# precomputed per-arm density grid in data/flies/behavior_density_chi_tau2s.npz.
bdn = np.load(f'{DATA_DIR}/behavior_density_chi_tau2s.npz')
cond = bdn['cond']                       # (4, ny, nx): chi-weighted density per arm
gx, gy = bdn['x_edges'], bdn['y_edges']
dev = cond - np.nanmean(cond, axis=0, keepdims=True)
print(f'behavior-map density grid: cond shape = {cond.shape}')

In [ ]:
arm_titles = ['Arm 1\nIdle & Slow', 'Arm 2\nAnterior Movements',
              'Arm 3\nPosterior & Wing Movements', 'Arm 4\nLocomotion']
vmax = float(np.nanpercentile(np.abs(dev), 99))
for j in range(4):
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    im = ax.pcolormesh(gx, gy, dev[j], cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                       shading='auto', rasterized=True)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(arm_titles[j], color=fg.ARM_PALETTE[j], fontsize=10)
    fg.save_panel(fig, f'fig5A_arm{j+1}_behavior_map')
    plt.show()

## Fig. 5B — apparent decay rate $r_k(\tau)$ vs lag

If the dynamics were Markovian at the chosen state resolution, each
$r_k(\tau) = -\log|\lambda_k(\tau)| / \tau$ would be constant in
$\tau$; instead, $r_2$ decreases ~120× over $\tau \in [0.01, 200]$ s.


In [ ]:
lags_seconds = np.array([0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0,
                          30.0, 60.0, 120.0, 200.0])
lags_frames = (lags_seconds * fs).astype(int)
r_k_full = np.zeros((len(lags_frames), 4))
for li, lg in enumerate(lags_frames):
    Tk = pp.make_transition_matrix(all_states, lag=int(lg), n_states=N)
    ev, _ = pp.leading_eigvecs(Tk, k=4)
    r_k_full[li] = -np.log(np.maximum(np.abs(ev[:4]), 1e-12)) / (lg / fs)

fig, ax = plt.subplots(figsize=(4.4, 3.0))
for k in range(4):
    ax.plot(lags_seconds, r_k_full[:, k], 'o-', lw=1.0,
             label=fr'$r_{{{k+2}}}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\tau$ (s)'); ax.set_ylabel(r'$r_k(\tau)$ (1/s)'); ax.legend()
plt.tight_layout()
fg.save_panel(fig, 'fig5B_r_k_vs_tau')
plt.show()


## Fig. 5C — predictive MI vs Markov prediction

We compare the empirical predictive mutual information
$I(\mathrm{arm}(t); \mathrm{arm}(t+\tau))$ between basin labels at two
times to the Markov prediction obtained by propagating
$T_\mathrm{basin}(\tau = 1~\mathrm{s})$ forward.


In [ ]:
def predictive_mi(seq, lag):
    seq = np.asarray(seq, dtype=int)
    M = int(seq.max()) + 1
    counts = np.zeros((M, M))
    np.add.at(counts, (seq[:-lag], seq[lag:]), 1.0)
    p = counts / counts.sum()
    pi = p.sum(axis=1); pj = p.sum(axis=0)
    mi = 0.0
    for i in range(M):
        for j in range(M):
            if p[i, j] > 0 and pi[i] > 0 and pj[j] > 0:
                mi += p[i, j] * np.log2(p[i, j] / (pi[i] * pj[j]))
    return mi

# Lump cluster sequence -> basin sequence.
basin_assign = chi.argmax(axis=1)
basin_seq = basin_assign[all_states]

# Empirical MI vs lag.
lag_grid = np.unique(np.r_[1, 2, 5, 10, 20, 50, 100, 200, 500,
                            1000, 2000, 5000, 10000])
mi_emp = np.array([predictive_mi(basin_seq, int(l * fs / 100)) for l in lag_grid])
# Markov prediction from T(tau=1s).
T1, pi1, _ = gu.lump_to_basin_msm(all_states, chi, lag=int(fs))
mi_markov = np.zeros_like(mi_emp, dtype=float)
for li, l in enumerate(lag_grid):
    n_steps = max(int(l), 1)
    Tn = np.linalg.matrix_power(T1, n_steps)
    p = pi1[:, None] * Tn
    pj = p.sum(axis=0)
    mi_v = 0.0
    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            if p[i, j] > 0 and pi1[i] > 0 and pj[j] > 0:
                mi_v += p[i, j] * np.log2(p[i, j] / (pi1[i] * pj[j]))
    mi_markov[li] = mi_v

fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.plot(lag_grid / fs, mi_emp, 'o-', color='C0', label='empirical')
ax.plot(lag_grid / fs, mi_markov, 's--', color='0.45',
         label=r'Markov from $T(1~\mathrm{s})$')
ax.set_xscale('log')
ax.set_xlabel(r'$\tau$ (s)'); ax.set_ylabel('predictive MI (bits)')
ax.legend()
plt.tight_layout()
fg.save_panel(fig, 'fig5C_predictive_mi')
plt.show()


## Fig. 5D — dwell-time CCDFs with truncated-power-law fits

The metastable residences are broad and heavy-tailed; each arm is best
described by a truncated power law $f(\tau)\propto\tau^{-\alpha}e^{-\lambda\tau}$
(Supp. Fig. S8 shows the full model comparison; S10/S11 show the
surrogate null and smoothing-scale robustness).

In [ ]:
# Fig. 5D: pooled per-arm metastable dwell-time CCDFs with truncated-power-law
# fits.  Residences use the canonical Delta=2 s definition
# (pp.metastable_residences); the truncated-PL exponents come out at
# alpha ~ 1.6, 1.7, 1.3, 1.3 for arms 1-4.
from powerlaw import Fit
fly_states_list = [states_per_fly[i] for i in sorted(states_per_fly)]
dwells = pp.metastable_residences(fly_states_list, chi, framerate=fs, delta=2.0)

fig, ax = plt.subplots(figsize=(4.8, 3.4))
for j in range(4):
    d = np.sort(dwells[j]); ccdf = 1 - np.arange(len(d)) / len(d)
    fit = Fit(d, discrete=False, verbose=False)
    a = fit.truncated_power_law.alpha
    lam = fit.truncated_power_law.parameter2
    ax.loglog(d[:-1], ccdf[:-1], color=fg.ARM_PALETTE[j], lw=1.6,
              label=fr'arm {j+1} ($\alpha={a:.1f}$)')
    xmin = fit.power_law.xmin; idx = np.searchsorted(d, xmin)
    if idx < len(d):
        c0 = 1 - idx / len(d)
        tau = np.logspace(np.log10(xmin), np.log10(d.max() * 1.2), 80)
        ftp = tau ** (-a) * np.exp(-lam * tau)
        cdf = np.cumsum(ftp[:-1] * np.diff(tau))
        if cdf[-1] > 0:
            ax.loglog(tau[:-1], c0 * (1 - cdf / cdf[-1]),
                      color=fg.ARM_PALETTE[j], lw=0.9, ls='--')
    print(f'arm {j+1}: alpha = {a:.2f}, 1/lambda = {1.0/lam:.0f} s')
ax.set_xlabel(r'metastable dwell time $\tau$ (s)')
ax.set_ylabel(r'CCDF $P(T \geq \tau)$')
ax.set_ylim(5e-4, 1.3)
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
fg.save_panel(fig, 'fig5D_dwell_ccdf')
plt.show()

## Fig. S7 — operator diagnostics + basin-count cross-validation

Parameter selection and spectral diagnostics for the fly operator.  The
panels that need the (un-shipped) wavelet PCs are gated behind
`RECOMPUTE_UPSTREAM`; the lag-sweep and leave-one-fly-out cross-validation
run from the shipped cluster sequences.  The full published panels are in
`figures/supp_figure_7.py`.

In [ ]:
# Fig. S7A: PCA spectrum vs shuffle threshold.  Needs the (un-shipped)
# wavelet PCs; the published panel is in figures/supp_figure_7.py.
if RECOMPUTE_UPSTREAM:
    proj_one, n_kept_one, eigvals_one, thresh_one = pp.pca_with_shuffle_threshold(
        np.abs(wlets_pca[0]), n_shuffles=5, max_keep=30, seed=0)
    fig, ax = plt.subplots(figsize=(4.0, 2.8))
    ax.plot(np.arange(1, len(eigvals_one) + 1), eigvals_one, 'o-', color='C0')
    ax.axhline(thresh_one, ls='--', color='r',
               label=f'shuffle threshold = {thresh_one:.2g}')
    ax.set_yscale('log'); ax.set_xlabel('PC index'); ax.set_ylabel('eigenvalue')
    ax.legend(); plt.tight_layout()
    fg.save_panel(fig, 'figS7A_pca_threshold'); plt.show()
else:
    print('Fig. S7A (PCA shuffle) needs the wavelet PCs; see figures/supp_figure_7.py')

In [ ]:
# Fig. S7B: Cao E_1(d) on the wavelet PCs.  Needs the (un-shipped) wavelet
# PCs; the published panel is in figures/supp_figure_7.py.
if RECOMPUTE_UPSTREAM:
    E1 = pp.cao_e1(all_pcs[:200000], max_d=20, tau=1, n_samples=10000, seed=0)
    fig, ax = plt.subplots(figsize=(4.0, 2.8))
    ax.plot(np.arange(2, len(E1) + 2), E1, 'o-')
    ax.axhline(1.0, ls=':', color='0.6')
    ax.set_xlabel(r'$d$'); ax.set_ylabel(r'$E_1(d)$')
    plt.tight_layout(); fg.save_panel(fig, 'figS7B_cao_E1'); plt.show()
else:
    print('Fig. S7B (Cao E_1) needs the wavelet PCs; see figures/supp_figure_7.py')

In [ ]:
# Fig. S7C: tau-sweep eigenvalues (stability of the leading modes across tau).
lags_s = np.array([0.5, 1.0, 2.0, 5.0, 10.0])
ev_by_tau = []
for ls_ in lags_s:
    Tt = pp.make_transition_matrix(all_states, lag=int(ls_ * fs), n_states=N)
    e, _ = pp.leading_eigvecs(Tt, k=6)
    ev_by_tau.append(np.abs(e[:6]))
ev_by_tau = np.array(ev_by_tau)
fig, ax = plt.subplots(figsize=(4.4, 3.0))
for k in range(5):
    ax.plot(lags_s, ev_by_tau[:, k], 'o-', label=fr'$|\lambda_{{{k+2}}}|$')
ax.set_xscale('log')
ax.set_xlabel(r'$\tau$ (s)'); ax.set_ylabel(r'$|\lambda_k|$'); ax.legend(ncol=2)
plt.tight_layout()
fg.save_panel(fig, 'figS7C_tau_sweep_eigvals')
plt.show()

In [ ]:
# Fig S7D: leave-one-fly-out CV of M.  The full version uses
# manuscript_plots/cv_basin_count.run_cv; here we just print a placeholder
# table from the cached chi.  See cv_basin_count.py in manuscript_plots/
# for the full implementation.
print('Per-fly chi reproducibility (cosines vs pooled):')
arm_dirs_pool = geo['arm_dirs']
cos_per_fly = []
for fly_id in sorted(states_per_fly):
    sf = states_per_fly[fly_id].astype(int)
    if len(sf) < lag * 4: continue

    # Restrict T to clusters this fly actually visited.
    visited = np.unique(sf)
    remap = -np.ones(N, dtype=int); remap[visited] = np.arange(len(visited))
    sf_r = remap[sf]
    Tf = pp.make_transition_matrix(sf_r, lag=lag, n_states=len(visited))
    pi_f = pp.stationary_distribution(Tf)
    try:
        out_f = gu.run_gpcca(Tf, M=4, eta=pi_f)
    except Exception:
        continue
    ev_f, vc_f = pp.leading_eigvecs(Tf, k=4)
    arms_f = gu.compute_hub_arms(vc_f[:, :3].real, pi_f, out_f['chi'])['arm_dirs']
    # 4x4 cosine -> Hungarian best match.
    from scipy.optimize import linear_sum_assignment
    C = -np.abs(arm_dirs_pool @ arms_f.T)
    row, col = linear_sum_assignment(C)
    cos_per_fly.append(-C[row, col])
cos_per_fly = np.array(cos_per_fly)
print(f'median per-arm cosine = {np.median(cos_per_fly, axis=0).round(3)}')
fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.boxplot(cos_per_fly, labels=[f'arm {j+1}' for j in range(4)])
ax.set_ylabel('per-fly arm cosine vs pooled')
plt.tight_layout()
fg.save_panel(fig, 'figS4D_per_fly_cosines')
plt.show()


---

All fly panels are saved as PNG/PDF under `outputs/`. The companion
`user_data.ipynb` lets you apply the same pipeline to a new $d$-dimensional
time series of your own choosing.


## Where to next

- **Read the biology.**  *Results Sec. "D. melanogaster"* — how the four arms
  map onto coarse behavioral categories and recover the long-range temporal
  memory documented at the action level (Berman 2016).
- **Canonical paper figures.**  `figures/figure_4.py`, `figures/figure_5.py`,
  and `figures/supp_figure_{7,8,9,10,11}.py` regenerate the published panels
  from the cached files in `data/flies/`.
- **Dwell-time tails.**  Supp. Figs. S8 (model selection + slow-mode shape),
  S10 (one-step-Markov surrogate null), and S11 (smoothing-scale robustness)
  characterize the broad, heavy-tailed metastable residences.
- **Apply this pipeline to your own data.**  Open `user_data.ipynb`.